In [ ]:
import pandas as pd
import numpy as np

# aggregated_results is produced by gencsv.py, which read all SUMMARY_* files and parse them into one CSV
# now contains 10 reps data for all 6 subjects, with ENHANCED/REDUCED results for FIX scenario, and deepseek-r1 results
data_path = "aggregated_results.csv"
data = pd.read_csv(data_path)
data.head()

In [ ]:
# check the unique values in the column
for column in data.columns:
    if column in ["file", "time"]:
        continue
    print(f"{column}: {data[column].unique()}")

In [ ]:
# get results for a specific config, like MSGONLY
data_msgonly = data[data['git_info'] == 'MSGONLY']
# count success of msgonly
data_msgonly_success = data_msgonly[data_msgonly['success'] == 1]
# unique by commit
data_msgonly_success_unique = data_msgonly_success.drop_duplicates(subset=['subject', 'commit'])
# group by scenario
data_msgonly_success_unique = data_msgonly_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_msgonly_success_unique)

### Table 2
Results for bug finding and bug reproduction effectiveness of Cleverest (default configuration)

In [ ]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# check the number of datapoints; expected to be 460 = 23 iid * 2 scenario * 10 repetition; if not, print warning
if len(data_def) != 460:
    print(
        f"!!Warning!!: the number of datapoints is {len(data_def)}; expected to be 460"
    )
data_def["reached"] = data_def["score"].apply(lambda x: 1 if x >= 1 else 0)
data_def["changed"] = data_def["score"].apply(lambda x: 1 if x >= 2 else 0)
# aggregate the data regarding the scenario and iid
data_def = (
    data_def.groupby(["scenario", "subject", "iid"])
    # .agg({"score": "mean", "success": "sum", "time": "mean"})
    .agg({"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"})
    .reset_index()
)
# scenario to multi columns, scenario first then (score, success, time)
data_def_pivot = data_def.pivot(index=["subject", "iid"], columns="scenario")
# order the column index
data_def_pivot = data_def_pivot.sort_index(level=1)
# add average row
average = data_def_pivot.agg({
    ("score", "BIC"): "mean",
    ("score", "FIX"): "mean",
    ("time", "BIC"): "mean",
    ("time", "FIX"): "mean",
    ("reached", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("reached", "FIX"): lambda x: (x != 0).sum(),
    ("changed", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("changed", "FIX"): lambda x: (x != 0).sum(),
    ("success", "BIC"): lambda x: (x != 0).sum(),  # Count non-zero values
    ("success", "FIX"): lambda x: (x != 0).sum(),
}).to_frame().T
average.index = pd.MultiIndex.from_tuples(
    [(r"\multicolumn{2}{c|}{Average}", "")]
)
data_def_pivot = pd.concat([data_def_pivot, average])
# check the column types
# display(data_def_pivot)
# sort data_def by scenario, and then iid
data_def = data_def.sort_values(by=["iid", "scenario"])
display(data_def)

Formatting for the latex table

In [ ]:
# order: iid
data_def_pivot_latex = data_def_pivot.copy()

###### index formats
# mujs -> \mujs
# libxml2 -> \libxml
# poppler -> \poppler
# jerryscript -> \jerryscript
# php-src -> \php
# z3 -> \zthree
##### also the level 1 should be "3" -> "\#3"
data_def_pivot_latex.index = data_def_pivot_latex.index.set_levels(
    [
        data_def_pivot_latex.index.levels[0].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                r"\multicolumn{2}{c|}{Average}": r"\multicolumn{2}{c|}{Average}",
            }
        ),
        data_def_pivot_latex.index.levels[1].map(lambda x: f"\\#{x}" if x else ""),
    ]
)

###### column formats
# score -> \SLIDER{3.3em}{score / 3:.2f}{circle}
data_def_pivot_latex["score"] = data_def_pivot_latex["score"].applymap(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
data_def_pivot_latex["reached"] = data_def_pivot_latex["reached"].applymap(
    lambda x: f"{int(x)}/10"
)
data_def_pivot_latex["changed"] = data_def_pivot_latex["changed"].applymap(
    lambda x: f"{int(x)}/10"
)
# success -> {success}/10
data_def_pivot_latex["success"] = data_def_pivot_latex["success"].applymap(
    lambda x: f"{int(x)}/10"
)
# time -> {time:.1f}
data_def_pivot_latex["time"] = data_def_pivot_latex["time"].applymap(
    lambda x: f"{x:.1f}"
)

# swap the column level
data_def_pivot_latex = data_def_pivot_latex.swaplevel(axis=1)
# sort the column level
data_def_pivot_latex = data_def_pivot_latex.sort_index(axis=1)
# change column order for to (BIC FIX) x (score reached changed success time)
data_def_pivot_latex = data_def_pivot_latex.reindex(
    columns=pd.MultiIndex.from_product(
        [["BIC", "FIX"], ["score", "reached", "changed", "success", "time"]]
    )
)

# show the columns
display(data_def_pivot_latex)


latex_str = data_def_pivot_latex.to_latex()
# remove [t] from multirow
latex_str = latex_str.replace("[t]", "").replace(
    r"\multicolumn{2}{c|}{Average} &  &",
    r"\multicolumn{2}{c|}{Average} &"
).replace(
    r"\cline{1-8}", r"\midrule"
).replace(
    r"\cline{1-12}", r"\midrule"
)



print(latex_str)
# print(data_def_pivot_latex.to_latex())

### Table 3
Results for our ablation study with average effectiveness score on a slider.

In [ ]:
config_prompt_msg = config_def.copy()
config_prompt_msg["git_info"] = "MSGONLY"
config_prompt_diff = config_def.copy()
config_prompt_diff["git_info"] = "DIFFONLY"
config_prompt_gen = config_def.copy()
config_prompt_gen["GENCMD"] = 1
config_llm_temp1 = config_def.copy()
config_llm_temp1["LLM_temp"] = 1
config_llm_mini = config_def.copy()
config_llm_mini["LLM"] = "gpt-4o-mini"
config_llm_deepseek = config_def.copy()
config_llm_deepseek["LLM"] = "deepseek-r1"
config_exec_iter10 = config_def.copy()
config_exec_iter10["max_iter"] = 10
config_exec_xfeedback = config_def.copy()
config_exec_xfeedback["NOFEEDBACK"] = 1

configs = {
    "default-": config_def,
    "prompt-msg": config_prompt_msg,
    "prompt-diff": config_prompt_diff,
    "prompt-gen": config_prompt_gen,
    "llm-temp1": config_llm_temp1,
    "llm-mini": config_llm_mini,
    "llm-deepseek": config_llm_deepseek,
    "exec-iter10": config_exec_iter10,
    "exec-xfeedback": config_exec_xfeedback,
}

data_each_config = []
for config_key, config in configs.items():
    data_config = data.copy()
    for key, value in config.items():
        if pd.isna(value):
            data_config = data_config[pd.isna(data_config[key])]
        else:
            data_config = data_config[data_config[key] == value]
    # check the number of datapoints for each config
    print(f"{config_key}: {len(data_config)}")
    # 12 commits in total, if len(data_config) != 460, print warning
    if len(data_config) != 460:
        if config_key == "prompt-gen" and len(data_config) == 140:
            pass  # 4 commits in mujs doesn't need GENCMD
        else:
            print(
                f"!!Warning!! {config_key} has {len(data_config)} data points; expected 460"
            )
    data_config = (
        data_config.groupby(["scenario", "subject", "iid"])
        .agg({"score": "mean"})
        .reset_index()
    )
    data_config["config_category"] = config_key.split("-")[0]
    data_config["config"] = config_key.split("-")[1]
    data_each_config.append(data_config)

data_each_config = pd.concat(data_each_config)
data_each_config = data_each_config.pivot(
    index=["scenario", "subject", "iid"],
    columns=["config_category", "config"],
    values="score",
)
# for the column ("prompt", "gen"), if the value is NaN, fill it with the value in ("default", "default")
data_each_config[("prompt", "gen")] = data_each_config[("prompt", "gen")].fillna(
    data_each_config[("default", "")]
)

# NOTE: FIX poppler 1305 (prompt, msg) data is wrong, if it is 0.2, manually change to 0.0
if data_each_config.loc[("FIX", "poppler", 1305), ("prompt", "msg")] == 0.2:
    data_each_config.loc[("FIX", "poppler", 1305), ("prompt", "msg")] = 0.0

dfs = []
# add average row for each scenario
for scenario in data_each_config.index.levels[0]:
    print(f"Scenario: {scenario}")
    # scenario is the first level of the index
    sub_data = data_each_config.loc[
        data_each_config.index.get_level_values(0) == scenario
    ]
    average = sub_data.agg("mean").to_frame().T
    average.index = pd.MultiIndex.from_tuples([(scenario, "Average", "")])
    # data_each_config = pd.concat([data_each_config, average])
    dfs = dfs + [sub_data, average]
# data_each_config = data_each_config.sort_index()
data_each_config = pd.concat(dfs)
# reorder the index level 1: ["mujs", "libxml2", "poppler", "jerryscript", "php-src", "z3", "Average"]
data_each_config.index = pd.MultiIndex.from_arrays([
    pd.Categorical(data_each_config.index.get_level_values(0), categories=[
        "BIC", "FIX"
    ], ordered=True),
    pd.Categorical(data_each_config.index.get_level_values(1), categories=[
        "mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "Average"
    ], ordered=True),
    data_each_config.index.get_level_values(2)
])
data_each_config = data_each_config.sort_index()
data_each_config

## gpt-4o-mini worse than default
# data_gpt4omini_worse = data_each_config[data_each_config[("llm", "mini")] < data_each_config[("default", "")]]
# data_gpt4omini_worse
# data_gpt4omini_worse_unreach = data_gpt4omini_worse[data_gpt4omini_worse[("llm", "mini")] == 0]
# data_gpt4omini_worse_unreach

## iter10 better than default
# data_iter10_better = data_each_config[data_each_config[("exec", "iter10")] < data_each_config[("default", "")]]
# data_iter10_better

## nofeedback worse than default
# data_nofeedback_worse = data_each_config[data_each_config[("exec", "xfeedback")] < data_each_config[("default", "")]]
# data_nofeedback_worse

In [ ]:
# to latex
data_each_config_latex = data_each_config.copy()
# index formats
data_each_config_latex.index = data_each_config_latex.index.set_levels(
    [
        data_each_config_latex.index.levels[0],
        data_each_config_latex.index.levels[1].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "Average": r"\multicolumn{2}{c|}{Average}",
            }
        ),
        data_each_config_latex.index.levels[2].map(
            lambda x: f"\\#{x}" if x else ""
        ),
    ]
)
###### column formats
# score -> \SLIDER{3.3em}{score / 3:.2f}{circle}
data_each_config_latex = data_each_config_latex.applymap(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
display(data_each_config_latex[("llm", "deepseek")])
# display(data_each_config_latex)
print(data_each_config_latex.to_latex().replace("[t]", "").replace(
    r"\multirow{24}{*}{BIC}", r"\multirow{24}{*}{\begin{sideways}Bug-finding\end{sideways}}"
).replace(
    r"\multirow{24}{*}{FIX}", r"\multirow{24}{*}{\begin{sideways}Bug-reproduction\end{sideways}}"
).replace(
    r"\multicolumn{2}{c|}{Average} &  &",
    r"\multicolumn{2}{c|}{Average} &"
).replace(
    r"\cline{1-12} \cline{2-12}", r"\midrule"
).replace(
    r"\cline{2-12}", r"\cmidrule{2-12}"
))

In [ ]:
# subset the data regarding the config
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]

data_def_success = data_def[data_def['success'] == 1]
data_def_success_unique = data_def_success.drop_duplicates(subset=['subject', 'commit'])
display(data_def_success_unique)
data_def_success_unique = data_def_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_def_success_unique)

data_r1 = data[data['LLM'] == 'deepseek-r1']
# count success of msgonly
data_r1_success = data_r1[data_r1['success'] == 1]
# unique by commit
data_r1_success_unique = data_r1_success.drop_duplicates(subset=['subject', 'commit'])
display(data_r1_success_unique)
# group by scenario
data_r1_success_unique = data_r1_success_unique.groupby(['scenario']).size().reset_index(name='count')
display(data_r1_success_unique)

Impact of iter 5 -> 10

In [ ]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# set index [scenario, subject, iid]
data_def = data_def.set_index(["scenario", "subject", "iid"])
# only FIX scenario
data_def = data_def.loc["FIX"]

config_def10 = {
    "git_info": "FULL",
    "max_iter": 10,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def10 = data.copy()
for key, value in config_def10.items():
    if pd.isna(value):
        data_def10 = data_def10[pd.isna(data_def10[key])]
    else:
        data_def10 = data_def10[data_def10[key] == value]
# set index [scenario, subject, iid]
data_def10 = data_def10.set_index(["scenario", "subject", "iid"])
# only FIX scenario
data_def10 = data_def10.loc["FIX"]

# merge the two dataframes
data_def = pd.merge(
    data_def,
    data_def10,
    on=["subject", "iid"],
    suffixes=("_5", "_10"),
)
# check the case where the score is different
# data_def_diff = data_def[
#     data_def["score_5"] != data_def["score_10"]
# ]
display(data_def[['score_5', 'score_10']])
# check the average for each (scenario, subject, iid)
data_def_diff_avg = data_def.groupby(["subject", "iid"]).agg(
    {"score_5": "mean", "score_10": "mean"}
)
display(data_def_diff_avg)
# count the number of 0, 1, 2, 3 in the score_5 and score_10 for each 
# (scenario, subject, iid)
data_def_diff_count = data_def.groupby(["subject", "iid"]).agg(
    {"score_5": "value_counts", "score_10": "value_counts"}
)
data_def_diff_count = data_def_diff_count.unstack()
# data_def_diff_count.columns = data_def_diff_count.columns.swaplevel(0, 1)
data_def_diff_count = data_def_diff_count.sort_index(axis=1)
data_def_diff_count

### (New RQ3): Impact of ENHANCED/REDUCED msg

In [ ]:
config_prompt_msg = config_def.copy()
config_prompt_msg["git_info"] = "MSGONLY"
config_prompt_diff = config_def.copy()
config_prompt_diff["git_info"] = "DIFFONLY"
config_prompt_enhanced = config_def.copy()
config_prompt_enhanced["git_info"] = "ENHANCED"
config_prompt_reduced = config_def.copy()
config_prompt_reduced["git_info"] = "REDUCED"


configs = {
    "default-": config_def,
    "prompt-msg": config_prompt_msg,
    "prompt-enhanced": config_prompt_enhanced,
    "prompt-reduced": config_prompt_reduced,
}

data_each_config = []
for config_key, config in configs.items():
    data_config = data.copy()
    for key, value in config.items():
        if pd.isna(value):
            data_config = data_config[pd.isna(data_config[key])]
        else:
            data_config = data_config[data_config[key] == value]
    data_config = data_config[data_config["scenario"] == "FIX"]
    # check the number of datapoints for each config
    print(f"{config_key}: {len(data_config)}")
    # 12 commits in total, if len(data_config) != 230, print warning
    if len(data_config) != 230:
        if config_key == "prompt-gen" and len(data_config) == 70:
            pass  # 4 commits in mujs doesn't need GENCMD
        else:
            print(
                f"!!Warning!! {config_key} has {len(data_config)} data points; expected 230"
            )
    data_config = (
        data_config.groupby(["scenario", "subject", "iid"])
        .agg({"score": "mean"})
        .reset_index()
    )
    data_config["config_category"] = config_key.split("-")[0]
    data_config["config"] = config_key.split("-")[1]
    data_each_config.append(data_config)

data_each_config = pd.concat(data_each_config)
data_each_config = data_each_config.pivot(
    index=["scenario", "subject", "iid"],
    columns=["config_category", "config"],
    values="score",
)

dfs = []
# add average row for each scenario
for scenario in data_each_config.index.levels[0]:
    print(f"Scenario: {scenario}")
    # scenario is the first level of the index
    sub_data = data_each_config.loc[
        data_each_config.index.get_level_values(0) == scenario
    ]
    average = sub_data.agg("mean").to_frame().T
    average.index = pd.MultiIndex.from_tuples([(scenario, "Average", "")])
    # data_each_config = pd.concat([data_each_config, average])
    dfs = dfs + [sub_data, average]
# data_each_config = data_each_config.sort_index()
data_each_config = pd.concat(dfs)
# reorder the index level 1: ["mujs", "libxml2", "poppler", "jerryscript", "php-src", "z3", "Average"]
data_each_config.index = pd.MultiIndex.from_arrays([
    pd.Categorical(data_each_config.index.get_level_values(0), categories=[
        "FIX"  # No BIC for enhanced/reduced
    ], ordered=True),
    pd.Categorical(data_each_config.index.get_level_values(1), categories=[
        "mujs", "libxml2", "poppler", "jerryscript", "z3", "php-src", "Average"
    ], ordered=True),
    data_each_config.index.get_level_values(2)
])
data_each_config = data_each_config.sort_index()
data_each_config

In [ ]:
# to latex
data_each_config_latex = data_each_config.copy()
# index formats
data_each_config_latex.index = data_each_config_latex.index.set_levels(
    [
        data_each_config_latex.index.levels[0],
        data_each_config_latex.index.levels[1].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "php-src": r"\php",
                "z3": r"\zthree",
                "Average": r"\multicolumn{2}{c|}{Average}",
            }
        ),
        data_each_config_latex.index.levels[2].map(
            lambda x: f"\\#{x}" if x else ""
        ),
    ]
)
###### column formats
# score -> \SLIDER{3.3em}{score / 3:.2f}{circle}
data_each_config_latex = data_each_config_latex.applymap(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
display(data_each_config_latex)
print(data_each_config_latex.to_latex().replace("[t]", "").replace(
    r"\multirow{24}{*}{BIC}", r"\multirow{24}{*}{\begin{sideways}Bug-finding\end{sideways}}"
).replace(
    r"\multirow{24}{*}{FIX}", r"\multirow{24}{*}{\begin{sideways}Bug-reproduction\end{sideways}}"
).replace(
    r"\multicolumn{2}{c|}{Average} &  &",
    r"\multicolumn{2}{c|}{Average} &"
).replace(
    r"\cline{1-11} \cline{2-11}", r"\midrule"
).replace(
    r"\cline{2-11}", r"\cmidrule{2-11}"
))

# Table 4

## WAFLGo results

In [ ]:
import glob

# init_result = [
#     ["BIC", "mujs", 65, "F"],
#     ["BIC", "mujs", 141, "F"],
#     ["BIC", "mujs", 145, "F"],
#     ["BIC", "mujs", 166, "B"],
#     ["BIC", "libxml2", 535, "B"],
#     ["BIC", "libxml2", 550, "R"],
#     ["BIC", "poppler", 1282, "R"],
#     ["BIC", "poppler", 1289, "R"],
#     ["BIC", "poppler", 1303, "F"],
#     ["BIC", "poppler", 1305, "F"],
#     ["BIC", "poppler", 1381, "R"],
#     ["FIX", "mujs", 65, "R"],
#     ["FIX", "mujs", 141, "F"],
#     ["FIX", "mujs", 145, "F"],
#     ["FIX", "mujs", 166, "B"],
#     ["FIX", "libxml2", 535, "B"],
#     ["FIX", "libxml2", 550, "F"],
#     ["FIX", "poppler", 1282, "R"],
#     ["FIX", "poppler", 1289, "R"],
#     ["FIX", "poppler", 1303, "F"],
#     ["FIX", "poppler", 1305, "R"],
#     ["FIX", "poppler", 1381, "R"],
# ]


def parse_init(data_path):
    data = pd.read_csv(data_path, sep=r"\s\|\s", engine="python")
    data["scenario"] = data_path.split("_")[-2]
    data["subject"] = data_path.split("_")[-3]
    data["init"] = data["final"].map(
        {"N": "F", "X": "F", "B": "B", "D": "D", "R": "R"}
    )
    data["iid"] = data["issue"]
    return data[["scenario", "subject", "iid", "init"]]


# parse_init("/home/nfs0/smbshares/softsec/seedscheck_mujs_BIC_withiid.csv")
init_result = pd.concat(
    [parse_init(f) for f in glob.glob("fuzz/seedscheck_*_withiid.csv")]
)
init_result.set_index(["scenario", "subject", "iid"], inplace=True)
init_result.sort_index(inplace=True)
init_result

In [ ]:
# parse WAFLGo result

# index = ["scenario", "iid"]
# columns = [["WAFLGo", "WAFLGo"], ["Result", "Time"]]

# Function to parse each waflgo_*.csv file
import glob


def parse_fuzz_csv(file_path):
    try:
        df = pd.read_csv(file_path, sep=r"\t", engine="python")
    except Exception as e:  # use different separator
        df = pd.read_csv(file_path, sep=r"\s\|\s", engine="python")
    print(f"Reading {file_path} with shape {df.shape}")
    df.columns = df.columns.str.strip()
    if 'scenario' not in df.columns:
        df["scenario"] = file_path.split(".")[0].split("_")[
            -2
        ]  # Extract scenario from file name
    if 'subject' not in df.columns:
        df["subject"] = file_path.split(".")[0].split("_")[
            1
        ]  # Extract subject from file name
    df["iid"] = df["issue"]
    df["success"] = df["final"].apply(
        lambda x: 1 if x == "B" else 0
    )  # Success if final is 'B'
    try:
        df["time"] = (
            df["time to find first crash in ms"].apply(
                pd.to_numeric, errors="coerce"
            )
            / 1000
        )  # Convert time to seconds
    except KeyError:  # postfuzz_*.csv
        df["time"] = (
            df["time to find first crash"].apply(pd.to_numeric, errors="coerce")
            / 1000
        )  # Convert time to seconds
    df.loc[df["success"] == 0, "time"] = (
        np.nan
    )  # some rows have time but not success, should be NaN
    # only keep columns: scenario, subject, iid, commit, idx, success, time
    df = df[["scenario", "subject", "iid", "commit", "idx", "success", "time"]]
    return df


# Parse and concatenate all CSV files
# csv_files = glob.glob("fuzz/waflgo_*_withiid.csv")
# waflgo_data = pd.concat([parse_fuzz_csv(file) for file in csv_files])
waflgo_data = parse_fuzz_csv("fuzz/merged_waflgo.csv")
display(waflgo_data.head())
# Group by scenario and iid, and aggregate success and time
waflgo_result = (
    waflgo_data.groupby(["scenario", "subject", "iid"])
    .agg(
        {
            "success": "sum",  # Number of successes
            "time": "mean",  # Average time in seconds
        }
    )
    .reset_index()
)

# Set index as scenario and iid
waflgo_result.set_index(["scenario", "subject", "iid"], inplace=True)

# Merge with initial results
waflgo_result = waflgo_result.join(init_result, how="outer")
# Reorder columns
waflgo_result = waflgo_result[["init", "success", "time"]]
# for the cases there init is B, set success 10
waflgo_result.loc[waflgo_result["init"] == "B", "success"] = 10
# for the cases there success is not NaN but time is NaN, set time 24 hours
waflgo_result.loc[
    waflgo_result["success"].notna() & waflgo_result["time"].isna(), "time"
] = 24 * 3600
# rename columns
waflgo_result.columns = pd.MultiIndex.from_product([["WAFLGo"], ["Init", "Bug", "Time"]])
waflgo_result

## Cleverest results

In [ ]:
# config for default
config_def = {
    "git_info": "FULL",
    "max_iter": 5,
    "LLM": "gpt-4o-2024-08-06",
    "LLM_temp": 0.5,
    "NOFEEDBACK": np.nan,  # choose rows with NaN
    "GENCMD": np.nan,
    # "RUNFUZZ": np.nan,
}
# subset the data regarding the confiG
data_def = data.copy()
for key, value in config_def.items():
    if pd.isna(value):
        data_def = data_def[pd.isna(data_def[key])]
    else:
        data_def = data_def[data_def[key] == value]
# check the number of datapoints; expected to 460 = 23 iid * 2 scenario * 10 repetition; if not, print warning
if len(data_def) != 460:
    print(
        f"!!Warning!!: the number of datapoints is {len(data_def)}; expected to be 460"
    )
# aggregate the data regarding the scenario and iid
data_def["reached"] = data_def["score"].apply(lambda x: 1 if x >= 1 else 0)
data_def["changed"] = data_def["score"].apply(lambda x: 1 if x >= 2 else 0)

data_def = (
    data_def.groupby(["scenario", "subject", "iid"])
    # .agg({"score": "mean", "success": "sum", "time": "mean"})
    .agg({"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"})
    .reset_index()
)

display(data_def.tail())
data_def_tb4 = data_def.copy()
data_def_tb4 = data_def_tb4.groupby(["scenario", "subject", "iid"]).agg(
    # {"score": "mean", "success": "sum", "time": "mean"}
    {"score": "mean", "reached": "sum", "changed": "sum", "success": "sum", "time": "mean"}
)
# rename columns
data_def_tb4.columns = pd.MultiIndex.from_product([["Cleverest"], ["Score", "Reach", "Change", "Bug", "Time"]])
data_def_tb4

## ClevFuzz results

In [ ]:
data_def[(data_def["subject"] == "poppler") & (data_def["scenario"] == "BIC") & (data_def["iid"] == 1282)]

In [ ]:
data_def = data.copy()
data_def

In [ ]:
data_def[(data_def["file"].str.contains("-1127"))]

In [ ]:
# Read all postfuzz_*.csv files
# csv_files = glob.glob("fuzz/postfuzz_*_withiid.csv")
# print(csv_files)
# csv_files = [f for f in csv_files if "withiid" not in f]
# fuzz_data = pd.concat([parse_fuzz_csv(file) for file in csv_files])
fuzz_data = parse_fuzz_csv("fuzz/merged_postfuzz.csv")
display(fuzz_data)

# for each row, find row from data_def with the 'file' includes the 'idx' in
# fuzz_data and 'iid' and 'commit' are the same; if not found, print error
clevfuzz_data = []
for idx, row in fuzz_data.iterrows():
    corr_row = data_def[
        (data_def["file"].str.contains(row["idx"]))
        & (data_def["iid"] == row["iid"])
        & (data_def["commit"] == row["commit"])
    ]
    if len(corr_row) != 1:
        print(f"Error: {len(corr_row)} rows found for {row['idx']}")
        # display full rows
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", None)
        pd.set_option("display.max_colwidth", None)
        display(corr_row)
        break
    # check if final result was either "R" or "B", if so, add to clevfuzz_data
    try:
        if (corr_row["final_result"].values[0] in ["R", "D"]) or (
            (
                corr_row[["scenario", "subject"]].values[0] == ("BIC", "mujs")
            ).all()
            and (corr_row["iid"].values[0] == 141)
        ):
            new_row = row.copy()
            new_row["cleverest"] = corr_row["final_result"].values[0]
            clevfuzz_data.append(new_row)
    except KeyError:
        display(corr_row)
        break
pd.reset_option("display.max_columns")
pd.reset_option("display.width")
pd.reset_option("display.max_colwidth")
clevfuzz_data = pd.DataFrame(clevfuzz_data)
display(clevfuzz_data)
clevfuzz_data.set_index(["scenario", "subject", "iid"], inplace=True)
clevfuzz_data.sort_index(inplace=True)
display(clevfuzz_data.loc[("BIC", "mujs", 141), :])
display(clevfuzz_data.loc[("BIC", "poppler", 1282), :])
display(clevfuzz_data.loc[("BIC", "mujs", 166), :])
display(clevfuzz_data.loc[("FIX", "mujs", 65), :])

# set 24 hours to the cases where time is NaN
clevfuzz_data.loc[clevfuzz_data["time"].isna(), "time"] = 24 * 60 * 60
# add row 'try' to count the number of tries
clevfuzz_data["try"] = 1
clevfuzz_data

In [ ]:
clevfuzz_result = clevfuzz_data.groupby(["scenario", "subject", "iid"]).agg(
    {"try": "sum", "success": "sum", "time": "mean"}
)
# bug: "{success}/{try}"
clevfuzz_result["bug"] = (
    clevfuzz_result["success"].astype(str)
    + "/"
    + clevfuzz_result["try"].astype(str)
)
clevfuzz_result = clevfuzz_result[["bug", "time", "success"]]
# rename columns
clevfuzz_result.columns = pd.MultiIndex.from_product(
    [["ClevFuzz"], ["Bug", "Time", "Success"]]
)
clevfuzz_result

In [ ]:
# merge waflgo_result, data_def_tb4, and clevfuzz_result
tb4 = waflgo_result.join(data_def_tb4, how="outer")
tb4 = tb4.join(clevfuzz_result, how="outer")
tb4.loc[:, ("ClevFuzz", "BugAll")] = tb4.apply(
    lambda x: (
        x["Cleverest"]["Bug"]
        if pd.isna(x["ClevFuzz"]["Bug"])
        else x["ClevFuzz"]["Success"] + x["Cleverest"]["Bug"]
    ),
    axis=1,
)
# drop ["ClevFuzz", "Success"]
tb4 = tb4.drop(("ClevFuzz", "Success"), axis=1)

# sort the index: ["Scenario", "Subject", "Issue"]
tb4.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4.index.get_level_values(1),
            categories=[
                "mujs",
                "libxml2",
                "poppler",
                "jerryscript",
                "z3",
                "php-src",
                "Aggregate",
            ],
            ordered=True,
        ),
        tb4.index.get_level_values(2),
    ],
    names=["Scenario", "Subject", "Issue"],
)
tb4 = tb4.sort_index()
tb4 = tb4[tb4["WAFLGo"]["Time"].notna()]  # keep only rows supported by WAFLGo
tb4

In [ ]:
# create the aggregation row
agg_rows = []
for scenario in tb4.index.levels[0]:
    print(f"Scenario: {scenario}")
    sub_data = tb4.loc[tb4.index.get_level_values(0) == scenario]
    len_sub_data = len(sub_data)
    # waflgo_bug aggregation: count the number of non-NaN or non-zero values
    waflgo_bug = (sub_data["WAFLGo"]["Bug"] > 0).sum()
    # waflgo_time aggregation: mean of non-NaN values
    col = sub_data["WAFLGo"]["Time"]
    col_nonnan = col[~col.isna()]
    waflgo_time = col_nonnan.mean()
    # cleverest_score aggregation: mean of values
    cleverest_score = sub_data["Cleverest"]["Score"].mean()
    # cleverest_bug aggregation: count the number of non-zero values
    cleverest_bug = (sub_data["Cleverest"]["Bug"] > 0).sum()
    cleverest_change = (sub_data["Cleverest"]["Change"] > 0).sum()
    cleverest_reach = (sub_data["Cleverest"]["Reach"] > 0).sum()
    # cleverest_time aggregation: mean of values
    cleverest_time = sub_data["Cleverest"]["Time"].mean()
    # clevfuzz_bugall aggregation: count the number of non-zero values
    clevfuzz_bugall = (sub_data["ClevFuzz"]["BugAll"] > 0).sum()
    agg_rows.append(
        {
            # ("Scenario", "Subject", "Issue"): (scenario, "Aggregate", ""),
            "Scenario": scenario,
            "Subject": "Aggregate",
            "Issue": "",
            ("WAFLGo", "Init", ""): "",
            ("WAFLGo", "Bug", ""): f"{waflgo_bug}/{len_sub_data}",
            ("WAFLGo", "Time", ""): str(
                pd.to_datetime(waflgo_time, unit="s").strftime("%H:%M:%S")
            ),
            (
                "Cleverest",
                "Score",
                "",
            ): f"\\SLIDER{{3.3em}}{{{(cleverest_score/3):.2f}}}{{circle}}",
            ("Cleverest", "Reach", ""): f"{cleverest_reach}/{len_sub_data}",
            ("Cleverest", "Change", ""): f"{cleverest_change}/{len_sub_data}",
            ("Cleverest", "Bug", ""): f"{cleverest_bug}/{len_sub_data}",
            ("Cleverest", "Time", ""): str(
                pd.to_datetime(cleverest_time, unit="s").strftime("%H:%M:%S")
            ),
            ("ClevFuzz", "Bug", ""): "",
            ("ClevFuzz", "Time", ""): "",
            ("ClevFuzz", "BugAll", ""): f"{clevfuzz_bugall}/{len_sub_data}",
        }
    )
agg_rows = pd.DataFrame(agg_rows)
# to multiindex
agg_rows.set_index(["Scenario", "Subject", "Issue"], inplace=True)
# display(agg_rows)
agg_rows.columns = pd.MultiIndex.from_tuples(agg_rows.columns)
display(agg_rows)

In [ ]:
# format
tb4_latex = tb4.copy()
## index formats
tb4_latex.index = tb4_latex.index.set_levels(
    [
        tb4_latex.index.levels[0],
        tb4_latex.index.levels[1].map(
            {
                "mujs": r"\mujs",
                "libxml2": r"\libxml",
                "poppler": r"\poppler",
                "jerryscript": r"\jerryscript",
                "z3": r"\zthree",
                "php-src": r"\php",
                "Aggregate": "Aggregate",
            }
        ),
        tb4_latex.index.levels[2].map(lambda x: f"\\#{x}" if x else ""),
    ],
)
tb4_latex.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4_latex.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(1),
            categories=[
                r"\mujs",
                r"\libxml",
                r"\poppler",
                r"\jerryscript",
                r"\zthree",
                r"\php",
                "Aggregate",
            ],
            ordered=True,
        ),
        tb4_latex.index.get_level_values(2),
    ],
    names=["Scenario", "Subject", "Issue"],
)
## column formats
### ["WAFLGo", "Init"]:
###  "F" -> r"\xmark",
###  "B" -> r"\cmark",
###  "R" -> r"\rmark",
###  NaN -> "-"
tb4_latex[("WAFLGo", "Init")] = tb4_latex[("WAFLGo", "Init")].map(
    {"F": r"\xmark", "B": r"\cmark", "R": r"\rmark", np.nan: "-"}
)
### ["WAFLGo", "Bug"]:
###  if number, "{int(x)}/10"
###  if NaN, "-"
tb4_latex[("WAFLGo", "Bug")] = tb4_latex[("WAFLGo", "Bug")].map(
    lambda x: f"{int(x)}/10" if not pd.isna(x) else "-"
)
### ["WAFLGo", "Time"]:
###  if NaN, "-"
###  if 86400, "T.O."
###  else, hh:mm:ss
tb4_latex[("WAFLGo", "Time")] = tb4_latex[("WAFLGo", "Time")].map(
    lambda x: (
        "-"
        if pd.isna(x)
        else (
            "T.O."
            if x == 86400
            else str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
        )
    )
)
### ["Cleverest", "Score"]:
###  f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
tb4_latex[("Cleverest", "Score")] = tb4_latex[("Cleverest", "Score")].map(
    lambda x: f"\\SLIDER{{3.3em}}{{{(x/3):.2f}}}{{circle}}"
)
### ["Cleverest", "Bug"]:
###  "{int(x)}/10"
tb4_latex[("Cleverest", "Reach")] = tb4_latex[("Cleverest", "Reach")].map(
    lambda x: f"{int(x)}/10"
)
tb4_latex[("Cleverest", "Change")] = tb4_latex[("Cleverest", "Change")].map(
    lambda x: f"{int(x)}/10"
)
tb4_latex[("Cleverest", "Bug")] = tb4_latex[("Cleverest", "Bug")].map(
    lambda x: f"{int(x)}/10"
)
### ["Cleverest", "Time"]:
###  hh:mm:ss
tb4_latex[("Cleverest", "Time")] = tb4_latex[("Cleverest", "Time")].map(
    lambda x: str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
)
### ["ClevFuzz", "Bug"]:
###  if NaN, "-/-"
###  else, as it is
tb4_latex[("ClevFuzz", "Bug")] = tb4_latex[("ClevFuzz", "Bug")].map(
    lambda x: x if not pd.isna(x) else "-/-"
)
### ["ClevFuzz", "Time"]:
###  if NaN, "-"
###  if 86400, "T.O."
###  else, hh:mm:ss
tb4_latex[("ClevFuzz", "Time")] = tb4_latex[("ClevFuzz", "Time")].map(
    lambda x: (
        "-"
        if pd.isna(x)
        else (
            "T.O."
            if x == 86400
            else str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
        )
    )
)
### ["ClevFuzz", "BugAll"]:
### "{int(x)}/10"
tb4_latex[("ClevFuzz", "BugAll")] = tb4_latex[("ClevFuzz", "BugAll")].map(
    lambda x: f"{int(x)}/10"
)

# Add the aggregation row
tb4_latex.loc[("BIC", r"\multicolumn{2}{c|}{Aggregate}", ""), :] = tuple(
    agg_rows.loc[("BIC", "Aggregate", "")]
)
tb4_latex.loc[("FIX", r"\multicolumn{2}{c|}{Aggregate}", ""), :] = tuple(
    agg_rows.loc[("FIX", "Aggregate", "")]
)

# sort the index
tb4_latex.index = pd.MultiIndex.from_arrays(
    [
        pd.Categorical(
            tb4_latex.index.get_level_values(0),
            categories=["BIC", "FIX"],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(1),
            categories=[
                r"\mujs",
                r"\libxml",
                r"\poppler",
                r"\jerryscript",
                r"\zthree",
                r"\php",
                r"\multicolumn{2}{c|}{Aggregate}",
            ],
            ordered=True,
        ),
        pd.Categorical(
            tb4_latex.index.get_level_values(2),
            categories=tb4_latex.index.get_level_values(2).unique(),
            ordered=True,
        ),
    ],
    names=["Scenario", "Subject", "Issue"],
)
tb4_latex = tb4_latex.sort_index()

display(tb4_latex)
tb4_latex = (
    tb4_latex.to_latex()
    .replace("[t]", "")
    .replace(
        r"\multirow{24}{*}{BIC}",
        r"\multirow{24}{*}{\begin{sideways}Bug-finding\end{sideways}}",
    )
    .replace(
        r"\multirow{24}{*}{FIX}",
        r"\multirow{24}{*}{\begin{sideways}Bug-reproduction\end{sideways}}",
    )
    .replace(
        r"\multirow{15}{*}{BIC}",
        r"\multirow{15}{*}{\begin{sideways}Bug-finding\end{sideways}}",
    )
    .replace(
        r"\multirow{15}{*}{FIX}",
        r"\multirow{15}{*}{\begin{sideways}Bug-reproduction\end{sideways}}",
    )
    .replace(
        r"\multicolumn{2}{c|}{Average} &  &", r"\multicolumn{2}{c|}{Average} &"
    )
    # .replace(r"\cline{1-12} \cline{2-12}", r"\midrule")
    # .replace(r"\cline{2-12}", r"\cmidrule{2-12}")
    .replace(r"\cline{1-14} \cline{2-14}", r"\midrule")
    .replace(r"\cline{2-14}", r"\cmidrule{2-14}")
    .replace(
        r"\multicolumn{2}{c|}{Aggregate} &  &",
        r"\multicolumn{2}{c|}{Aggregate} &",
    )
)
print(tb4_latex)

# Figure 3

In [ ]:
clevfuzz_data[["cleverest", "success"]]

data = {}
for scenario in ["BIC", "FIX"]:
    sub_data = clevfuzz_data.loc[scenario]
    # count the number of R in cleverest and 1 in success
    R_B = len(
        sub_data[
            (sub_data["cleverest"] == "R") & (sub_data["success"] == 1)
        ]
    )
    D_B = len(
        sub_data[
            (sub_data["cleverest"] == "D") & (sub_data["success"] == 1)
        ]
    )
    R_F = len(
        sub_data[
            (sub_data["cleverest"] == "R") & (sub_data["success"] == 0)
        ]
    )
    D_F = len(
        sub_data[
            (sub_data["cleverest"] == "D") & (sub_data["success"] == 0)
        ]
    )
    data[scenario] = {
        "R_B": R_B,
        "D_B": D_B,
        "R_F": R_F,
        "D_F": D_F,
    }
clevfuzz_convert = pd.DataFrame(data).T
clevfuzz_convert["B"] = clevfuzz_convert["R_B"] + clevfuzz_convert["D_B"]
clevfuzz_convert["F"] = clevfuzz_convert["R_F"] + clevfuzz_convert["D_F"]
# column order = ["R_B", "D_B", "B", "R_F", "D_F", "F"]
clevfuzz_convert = clevfuzz_convert[
    ["R_B", "D_B", "B", "R_F", "D_F", "F"]
]
clevfuzz_convert.loc["sum"] = clevfuzz_convert.sum()
clevfuzz_convert

In [ ]:
tb4_time = tb4.copy()
## only rows where WAFLGo time is not NaN
tb4_time = tb4_time[tb4_time["WAFLGo"]["Time"].notna()]
## total time = cleverest time if clevfuzz time is NaN, else sum of cleverest
## and clevfuzz time
tb4_time["ClevFuzz", "TotalTime"] = tb4_time["Cleverest"]["Time"]
tb4_time.loc[
    tb4_time["ClevFuzz"]["Time"].notna(), ("ClevFuzz", "TotalTime")
] = (tb4_time["Cleverest"]["Time"] + tb4_time["ClevFuzz"]["Time"])
display(tb4_time)
# average time for each scenario
tb4_time_avg = tb4_time.groupby("Scenario").agg(
    {("WAFLGo", "Time"): "mean", ("ClevFuzz", "TotalTime"): "mean"}
)
# format to hh:mm:ss
tb4_time_avg = tb4_time_avg.applymap(
    lambda x: str(pd.to_datetime(x, unit="s").strftime("%H:%M:%S"))
)
tb4_time_avg

# Comment statistics

In [ ]:
# read columns msgwords,msgchars,difflines,diffchars from stats_22commits.csv, insert into data_each_config
# there are already "scenario" and "iid" columns in the stats_22commits.csv, so we can merge them
stats_path = "../stat_22commits.csv"
stats = pd.read_csv(stats_path)
# flatten multi-level column index
data_stats = data_each_config.copy()
# data_stats = data_stats.reset_index()
data_stats.columns = data_stats.columns.get_level_values(1)
data_stats = data_stats.reset_index()
# rename data_stats[2] column to data_stats['default]
data_stats = data_stats.rename(columns={data_stats.columns[2]: "default"})
print(data_stats.columns)
data_stats = data_stats[["scenario", "iid", "default", "msg", "diff"]]
data_stats_merged = data_stats.merge(stats, on=["scenario", "iid"])
# # set index back
data_stats_merged
# print as csv
data_stats_merged.to_csv("data_stats_merged.csv", index=False)